In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, WeightedRandomSampler

import internal.utils.telegram_utils as tg
from internal.data_types import PainDataset
from internal.nn.models.pain_tcn_bilstm_attn import PainTCNBiLSTMAttn
from internal.nn.training.pain_tcn_bilstm_attn_trainer import PainTCNBiLSTMAttnTrainer
from internal.persistence_manager import PersistenceManager

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
@torch.no_grad()
def infer_logits(_model, _x_num, _x_surv, _x_sta, _x_summ, batch_size=64):
    _model.eval()
    ds = PainDataset(_x_num, _x_surv, _x_sta, _x_summ)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    predictions = []
    for b in dl:
        x_num  = b['x_num'].to(device)
        x_sta  = b['x_sta'].to(device)
        x_surv = [s.to(device) for s in b['x_surv']]
        x_summ = b['x_summ'].to(device)
        logits = _model(x_num, x_surv, x_sta, x_summ)  # (B,3)
        predictions.append(logits.detach().cpu().numpy())
    return np.concatenate(predictions, axis=0)  # (N,3)

@torch.no_grad()
def infer_logits_tta(_model, _x_num, _x_surv, _x_sta, _x_summ, _n_aug=5, _noise_std=0.01, _roll=2) -> np.ndarray:
    _model.eval()
    outs = []
    for _k in range(_n_aug):
        xn = _x_num.copy()
        if _k > 0:
            # small Gaussian jitter on numeric channels
            xn += np.random.normal(0, _noise_std, size=xn.shape).astype(xn.dtype)
            # tiny circular time shift
            shift = np.random.randint(-_roll, _roll + 1)
            if shift != 0:
                xn = np.roll(xn, shift=shift, axis=1)  # roll along the time
        outs.append(infer_logits(_model, xn, _x_surv, _x_sta, _x_summ, batch_size=64))
    return np.mean(outs, axis=0)  # (N,3)

In [3]:
# load data
data = PersistenceManager.load_arrays_v2()

# set the seed for reproducibility
seed = 2021
np.random.seed(seed)
torch.manual_seed(seed)

# prepare stratified group K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
pid_all     = np.concatenate([data.pid_train,        data.pid_val], axis=0)
y_all       = np.concatenate([data.y_train,          data.y_val],   axis=0)
X_num_all   = np.concatenate([data.X_dyn_num_train,  data.X_dyn_num_val],  axis=0)
X_sta_all   = np.concatenate([data.X_sta_train,      data.X_sta_val],      axis=0)
X_surv_all  = [np.concatenate([data.X_surv_train[i], data.X_surv_val[i]], axis=0) for i in range(4)]
X_summ_all  = np.concatenate([data.X_dyn_summ_train, data.X_dyn_summ_val], axis=0)

# range of class-2 multipliers to try
class2_multipliers = [1, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3]
# to store per-multiplier CV stats
mult_stats: dict[float, dict[str, float]] = {}

# trainer instance
trainer = PainTCNBiLSTMAttnTrainer(
    classes=[0, 1, 2],
    epochs=1000,
    patience=28,
    warmup_epochs=20,
    p_drop_summ=0.3,
    summary_window=data.feat_eng.window,
    device=device
)

# notify me about phase A start
tg.send_message("🚀 Phase A: Searching best class-2 multiplier via CV...", raise_on_failure=False)

# for each class-2 multiplier, do a full CV and record mean/std
for class2_multiplier in class2_multipliers:
    # to store fold scores for this multiplier
    fold_scores_m = []

    # K-Fold CV loop - no test logits here
    for k, (tr_idx, va_idx) in enumerate(skf.split(X=np.zeros(len(y_all)),
                                                   y=y_all,
                                                   groups=pid_all)):
        print(f"=== [Phase A] Fold {k} (grouped by patient) with class-2 multiplier {class2_multiplier} ===")

        # --- prepare datasets ---
        Xtr_num, Xva_num    = X_num_all[tr_idx], X_num_all[va_idx]
        Xtr_sta, Xva_sta    = X_sta_all[tr_idx], X_sta_all[va_idx]
        Xtr_surv            = [s[tr_idx] for s in X_surv_all]
        Xva_surv            = [s[va_idx] for s in X_surv_all]
        Xtr_summ, Xva_summ  = X_summ_all[tr_idx], X_summ_all[va_idx]
        ytr, yva            = y_all[tr_idx],      y_all[va_idx]
        pid_tr, pid_va      = pid_all[tr_idx],    pid_all[va_idx]

        # --- sanity check: no leakage ---
        assert set(pid_tr).isdisjoint(set(pid_va)), "Patient leaked between train and val!"

        # --- create datasets ---
        train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
        val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

        # --- create weighted sampler for class balancing ---
        class_counts = np.bincount(ytr, minlength=3).astype(np.float32)
        inv = 1.0 / np.clip(class_counts, 1, None)
        inv = inv / inv.mean()
        alpha = 0.5
        w_class = np.power(inv, alpha)
        cap = 5.0
        w_class = np.minimum(w_class, cap * w_class.mean())
        sample_w = w_class[ytr]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_w, dtype=torch.double),
            num_samples=len(ytr),
            replacement=True
        )

        # --- create data loaders that use the sampler ---
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, sampler=sampler, num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

        # -- calculate priors from training fold ---
        counts_tr = np.bincount(ytr, minlength=3).astype(np.float32)
        priors_tr = counts_tr / counts_tr.sum()

        # --- Train on this fold ---
        model, f1, best_b, t_fold, best_tau = trainer.train_one_fold(
            model=PainTCNBiLSTMAttn(
                d_num=90,
                d_emb=4,
                d_sta=7,
                d_summ=data.X_dyn_summ_train.shape[1],
                tcn_channels=128,
                lstm_hidden=128,
                num_classes=3,
                dropout=0.3
            ).to(device),
            k=k,
            train_loader=train_loader,
            val_loader=val_loader,
            y_train=ytr,
            y_val=yva,
            priors=priors_tr,
            class_multipliers={2: class2_multiplier}
        )

        # record fold score
        fold_scores_m.append(f1)

    # after all folds, compute mean/std for this multiplier
    mean_m = float(np.mean(fold_scores_m))
    std_m  = float(np.std(fold_scores_m))
    mult_stats[class2_multiplier] = {'mean': mean_m, 'std': std_m}

    # notify me about this multiplier's CV result
    print(f"[Phase A] mul={class2_multiplier}: mean={mean_m:.4f} std={std_m:.4f}")
    tg.send_message(
        f"📊 [Phase A] class2_multiplier={class2_multiplier} → mean F1={mean_m:.4f}, std={std_m:.4f}",
        raise_on_failure=False
    )

# after all multipliers, print summary
print("=== [Phase A] Class-2 Multiplier Search Complete ===")
print("Per-multiplier CV:")
for m in class2_multipliers:
    print(f"  mul={m}: mean={mult_stats[m]['mean']:.4f} std={mult_stats[m]['std']:.4f}")

best_mult = max(mult_stats.keys(), key=lambda m: mult_stats[m]['mean'])
print("Selected best multiplier:", best_mult, "with mean F1:", mult_stats[best_mult]['mean'])
tg.send_message(
    f"✅ [Phase A] Best class2_multiplier={best_mult} with mean F1={mult_stats[best_mult]['mean']:.4f}",
    raise_on_failure=False
)

Arrays v2 loaded successfully from: /home/andre/university/AN2DL-Challenge-1/notebooks/processed/arrays_v2.joblib
=== [Phase A] Fold 0 (grouped by patient) with class-2 multiplier 1 ===
[F0 001] t_loss=0.4206 | F1(macro)=0.4598 | rec=[0.873 0.4   0.1  ] | lr=1.00e-03 | patience=1/28
[F0 002] t_loss=0.3288 | F1(macro)=0.5887 | rec=[0.951 0.2   0.8  ] | lr=1.00e-03 | patience=1/28
[F0 003] t_loss=0.2743 | F1(macro)=0.6841 | rec=[0.941 0.8   0.3  ] | lr=1.00e-03 | patience=1/28
[F0 004] t_loss=0.2293 | F1(macro)=0.8131 | rec=[0.971 0.7   0.8  ] | lr=1.00e-03 | patience=1/28
[F0 005] t_loss=0.1848 | F1(macro)=0.7661 | rec=[0.912 0.9   0.6  ] | lr=1.00e-03 | patience=1/28
[F0 006] t_loss=0.1361 | F1(macro)=0.8412 | rec=[0.98 0.75 0.8 ] | lr=1.00e-03 | patience=2/28
[F0 007] t_loss=0.1077 | F1(macro)=0.8885 | rec=[0.98 0.85 0.8 ] | lr=1.00e-03 | patience=1/28
[F0 008] t_loss=0.0952 | F1(macro)=0.8595 | rec=[0.951 0.8   0.9  ] | lr=1.00e-03 | patience=1/28
[F0 009] t_loss=0.0893 | F1(macro)=0

{}

In [4]:
# --- Phase B: Final training with the best multiplier and test ensembling ---

# to store test logits from each fold
test_fold_logits = []
# to store final fold scores
fold_scores_final = []
# to store temperatures within final folds
temperatures = []
# to store taus within final folds
taus = []

# notify me about phase B start
tg.send_message(
    f"🚀 Phase B: Final training with best class2_multiplier={best_mult} and test ensembling...",
    raise_on_failure=False
)

# K-Fold CV loop with best multiplier and test logits
for k, (tr_idx, va_idx) in enumerate(skf.split(X=np.zeros(len(y_all)),
                                               y=y_all,
                                               groups=pid_all)):
    print(f"=== [Phase B] Fold {k} (grouped by patient) with class-2 multiplier {best_mult} ===")

    # --- prepare datasets ---
    Xtr_num, Xva_num    = X_num_all[tr_idx], X_num_all[va_idx]
    Xtr_sta, Xva_sta    = X_sta_all[tr_idx], X_sta_all[va_idx]
    Xtr_surv            = [s[tr_idx] for s in X_surv_all]
    Xva_surv            = [s[va_idx] for s in X_surv_all]
    Xtr_summ, Xva_summ  = X_summ_all[tr_idx], X_summ_all[va_idx]
    ytr, yva            = y_all[tr_idx],      y_all[va_idx]
    pid_tr, pid_va      = pid_all[tr_idx],    pid_all[va_idx]

    # --- sanity check: no leakage ---
    assert set(pid_tr).isdisjoint(set(pid_va)), "Patient leaked between train and val!"

    # --- create datasets ---
    train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
    val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

    # --- create weighted sampler for class balancing ---
    class_counts = np.bincount(ytr, minlength=3).astype(np.float32)
    inv = 1.0 / np.clip(class_counts, 1, None)
    inv = inv / inv.mean()
    alpha = 0.5
    w_class = np.power(inv, alpha)
    cap = 5.0
    w_class = np.minimum(w_class, cap * w_class.mean())
    sample_w = w_class[ytr]
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.double),
        num_samples=len(ytr),
        replacement=True
    )

    # --- create data loaders that use the sampler ---
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    # -- calculate priors from training fold ---
    counts_tr = np.bincount(ytr, minlength=3).astype(np.float32)
    priors_tr = counts_tr / counts_tr.sum()

    # --- Train on this fold ---
    model, f1, best_b, t_fold, best_tau = trainer.train_one_fold(
        model=PainTCNBiLSTMAttn(
            d_num=90,
            d_emb=4,
            d_sta=7,
            d_summ=data.X_dyn_summ_train.shape[1],
            tcn_channels=128,
            lstm_hidden=128,
            num_classes=3,
            dropout=0.3
        ).to(device),
        k=k,
        train_loader=train_loader,
        val_loader=val_loader,
        y_train=ytr,
        y_val=yva,
        priors=priors_tr,
        class_multipliers={2: best_mult}
    )

    # --- record fold score ---
    fold_scores_final.append(f1)
    temperatures.append(t_fold)
    taus.append(best_tau)

    # --- infer test logits with TTA ---
    logits_te = infer_logits_tta(
        model,
        data.X_dyn_num_test,
        data.X_surv_test,
        data.X_sta_test,
        data.X_dyn_summ_test,
        _n_aug=5, _noise_std=0.01, _roll=2
    ).astype(np.float32)

    # --- temperature scaling (per-fold scalar) ---
    logits_te = logits_te / float(t_fold)

    # -- prior correction with the best tau and bias ---
    log_p = np.log(np.clip(priors_tr, 1e-8, 1.0)).astype(np.float32)
    logits_te = logits_te + best_tau * log_p[None, :]
    logits_te = logits_te + best_b[None, :]

    # --- store for ensembling ---
    test_fold_logits.append(logits_te)

# --- ensemble and submission ---
logits_stack = np.stack(test_fold_logits, axis=0)   # (5, N_test, 3)
logits_avg   = logits_stack.mean(axis=0)            # (N_test, 3)
pred_idx     = logits_avg.argmax(axis=1)

idx_to_label = {0:'no_pain', 1:'low_pain', 2:'high_pain'}
pred_labels  = [idx_to_label[int(i)] for i in pred_idx]

ids_test = [f'{i:03}' for i in range(len(pred_labels))]
sub = pd.DataFrame({'sample_index': ids_test, 'label': pred_labels})

_filename = f'PainTCNBiLSTMAttn_bestmul_{best_mult}_seed{seed}'
sub.to_csv(f'submissions/{_filename}.csv', index=False)
print("Saved:", f'submissions/{_filename}.csv')
tg.send_file(
    f'submissions/{_filename}.csv',
    caption=f"✅ Submission from best_mult={best_mult}",
    raise_on_failure=False
)

print(f"[Phase B] Final CV with best_mult={best_mult}: mean F1={np.mean(fold_scores_final):.4f} std={np.std(fold_scores_final):.4f}")
tg.send_message(
    f"✅ [Phase B] Done. CV mean={np.mean(fold_scores_final):.4f} std={np.std(fold_scores_final):.4f}",
    False
)


=== [Phase B] Fold 0 (grouped by patient) with class-2 multiplier 1 ===
[F0 001] t_loss=0.4246 | F1(macro)=0.3994 | rec=[0.882 0.05  0.4  ] | lr=1.00e-03 | patience=1/28
[F0 002] t_loss=0.3598 | F1(macro)=0.4421 | rec=[0.98 0.2  0.1 ] | lr=1.00e-03 | patience=1/28
[F0 003] t_loss=0.2803 | F1(macro)=0.6203 | rec=[0.98 0.55 0.3 ] | lr=1.00e-03 | patience=1/28
[F0 004] t_loss=0.2105 | F1(macro)=0.7189 | rec=[0.873 0.85  0.6  ] | lr=1.00e-03 | patience=1/28
[F0 005] t_loss=0.1619 | F1(macro)=0.7671 | rec=[0.961 0.75  0.6  ] | lr=1.00e-03 | patience=1/28
[F0 006] t_loss=0.1122 | F1(macro)=0.7704 | rec=[0.961 0.7   0.7  ] | lr=1.00e-03 | patience=1/28
[F0 007] t_loss=0.0794 | F1(macro)=0.8598 | rec=[0.98 0.85 0.7 ] | lr=1.00e-03 | patience=1/28
[F0 008] t_loss=0.0574 | F1(macro)=0.8918 | rec=[0.99 0.85 0.8 ] | lr=1.00e-03 | patience=1/28
[F0 009] t_loss=0.0517 | F1(macro)=0.7890 | rec=[0.941 0.75  0.7  ] | lr=1.00e-03 | patience=1/28
[F0 010] t_loss=0.0302 | F1(macro)=0.8241 | rec=[0.951 0.8

{'ok': True,
 'result': {'message_id': 6276,
  'from': {'id': 8015888206,
   'is_bot': True,
   'first_name': 'Track-Devices',
   'username': 'track_devices_bot'},
  'chat': {'id': 205313466,
   'first_name': 'Andre',
   'username': 'AndreVale69',
   'type': 'private'},
  'date': 1763029533,
  'text': '✅ Phase B Done. CV mean=0.8144 std=0.0617'}}